# LC 986 — Interval List Intersections
**Difficulty:** Medium &nbsp;|&nbsp; **Category:** Intervals
**Pattern:** Two Pointers — Advance the Pointer with
the Smaller End

<div style="border-left:4px solid purple; padding:10px 16px;
            background:#f5f0ff; margin-top:12px;">
<strong>Core Insight:</strong> Use two pointers,
one per list. The overlap of current pair is
[max(starts), min(ends)] — valid if start <= end.
Always advance the pointer whose interval ends
earlier, as it cannot overlap any future interval
in the other list.
</div>

## Official Problem Statement

You are given two lists of closed intervals,
`firstList` and `secondList`, where
`firstList[i] = [starti, endi]` and
`secondList[j] = [startj, endj]`.

Each list of intervals is pairwise disjoint and
in sorted order.

Return the **intersection** of these two interval
lists. A **closed interval** `[a, b]` (with
`a <= b`) denotes the set of real numbers `x`
with `a <= x <= b`.

The intersection of two closed intervals is a set
of real numbers that are either empty or
represented as a closed interval.

**Example 1:**
```
Input:
  firstList  = [[0,2],[5,10],[13,23],[24,25]]
  secondList = [[1,5],[8,12],[15,24],[25,26]]
Output:
  [[1,2],[5,5],[8,10],[15,23],[24,24],[25,25]]
```
**Example 2:**
```
Input:  firstList = [[1,3],[5,9]], secondList = []
Output: []
```

**Constraints:**
- `0 <= firstList.length, secondList.length <= 1000`
- `firstList[i].length == secondList[j].length == 2`
- `0 <= starti <= endi <= 10^9`
- `endi < start(i+1)` — each list is disjoint
- `0 <= startj <= endj <= 10^9`

## What This Is Actually Asking

You have two separate sorted lists of time ranges.
Find every time range where both lists are active
simultaneously. Return those shared ranges. Both
lists are already sorted and have no internal
overlaps.

## Walk Through an Example by Hand

```
A = [[0,2],[5,10],[13,23],[24,25]]
B = [[1,5],[8,12],[15,24],[25,26]]
i=0  j=0

i=0 j=0: A=[0,2]  B=[1,5]
  overlap = [max(0,1), min(2,5)] = [1,2]  valid -> add
  A ends earlier (2 < 5) -> advance i

i=1 j=0: A=[5,10] B=[1,5]
  overlap = [max(5,1), min(10,5)] = [5,5]  valid -> add
  B ends earlier (5 <= 10) -> advance j

i=1 j=1: A=[5,10] B=[8,12]
  overlap = [max(5,8), min(10,12)] = [8,10]  valid -> add
  A ends earlier (10 < 12) -> advance i

i=2 j=1: A=[13,23] B=[8,12]
  overlap = [max(13,8), min(23,12)] = [13,12]  invalid
  B ends earlier (12 < 23) -> advance j

i=2 j=2: A=[13,23] B=[15,24]
  overlap = [max(13,15), min(23,24)] = [15,23]  valid -> add
  A ends earlier (23 < 24) -> advance i

i=3 j=2: A=[24,25] B=[15,24]
  overlap = [max(24,15), min(25,24)] = [24,24]  valid -> add
  B ends earlier (24 <= 25) -> advance j

i=3 j=3: A=[24,25] B=[25,26]
  overlap = [max(24,25), min(25,26)] = [25,25]  valid -> add
  both end same (25==25) -> advance i

i=4 -> done
Result: [[1,2],[5,5],[8,10],[15,23],[24,24],[25,25]]
```

## The Picture

```
A: [0--2]     [5-----10]   [13---------23] [24-25]
B:    [1--5]       [8--12]    [15--------24]  [25-26]

Overlaps (both active at same time):
     [1,2] [5] [8,10]  [15,23] [24] [25]

Two-pointer decision at each step:

  lo = max(A.start, B.start)  <- overlap start
  hi = min(A.end,   B.end)    <- overlap end

  lo <= hi? -> valid overlap -> add [lo, hi]
  lo >  hi? -> no overlap    -> skip

  Advance: whichever interval ends first
  because it cannot overlap any LATER interval
  in the other list (the other list is sorted).

  A.end <= B.end -> i += 1
  else           -> j += 1
```

## When To Use This Pattern

- When finding overlaps between **two sorted
  disjoint lists**, think **two pointers**
- When computing an overlap, think
  **[max(starts), min(ends)] — valid if start<=end**
- When deciding which pointer to advance, think
  **advance the one with the smaller end**
- When one list is exhausted, think
  **loop ends — no more overlaps possible**

## The Approach

Set two pointers i and j starting at zero.
While both lists have elements, compute the overlap
as [max of starts, min of ends]. If the overlap
start is at most the end, it is valid — add it to
the result.
Then advance the pointer whose current interval
ends first. Return the collected intersections.

In [ ]:
from typing import List  # type hints for the solution

In [ ]:
def test_harness(func):
    tests = [
        # (firstList, secondList, expected)
        (
            [[0,2],[5,10],[13,23],[24,25]],
            [[1,5],[8,12],[15,24],[25,26]],
            [[1,2],[5,5],[8,10],[15,23],[24,24],[25,25]]
        ),
        ([[1,3],[5,9]], [],       []),
        ([],            [[1,3]],  []),
        ([[1,7]],       [[3,5]],  [[3,5]]),  # A contains B
        ([[3,5]],       [[1,7]],  [[3,5]]),  # B contains A
        ([[1,2],[3,4]], [[1,4]],  [[1,2],[3,4]]),
        ([[0,0],[2,2]], [[1,1]],  []),       # no overlap
    ]

    passed = 0
    for i, (A, B, expected) in enumerate(tests):
        result = func([x[:] for x in A], [x[:] for x in B])
        ok = result == expected
        status = "PASSED" if ok else "FAILED"
        if ok:
            passed += 1
        print(
            f"Test {i+1}: {status} | "
            f"expected={expected} | got={result}"
        )

    print(f"\n{passed}/{len(tests)} tests passed")

In [ ]:
def intervalIntersection(
    firstList: List[List[int]],
    secondList: List[List[int]]
) -> List[List[int]]:
    """
    Return all intersections of two sorted interval lists.

    Two pointers i, j. At each step: overlap =
    [max(A[i][0],B[j][0]), min(A[i][1],B[j][1])]. If
    valid (lo<=hi) add it. Advance the pointer whose
    interval ends earlier.

    Time:  O(m+n) — each interval visited at most once
    Space: O(m+n) — output list in worst case
    """
    pass


# Quick debug — run this cell while building
A = [[0,2],[5,10],[13,23],[24,25]]
B = [[1,5],[8,12],[15,24],[25,26]]
print(intervalIntersection(A, B))
# [[1,2],[5,5],[8,10],[15,23],[24,24],[25,25]]
print(intervalIntersection([[1,3],[5,9]], []))  # []
print(intervalIntersection([[1,7]], [[3,5]]))   # [[3,5]]

In [ ]:
# Uncomment and run when solution is ready
# test_harness(intervalIntersection)

## Complexity

| Approach | Time | Space |
|---|---|---|
| Brute force — all pairs | O(m × n) | O(1) |
| Two pointers | O(m + n) | O(m + n) |

Because both lists are sorted and disjoint, the
two-pointer approach visits each interval exactly
once — total work is proportional to the combined
length of both lists.

## Real World Connection

At Citi, two teams each produce sorted lists of
maintenance windows for the same server cluster.
Finding the times when BOTH teams are in maintenance
simultaneously — the true blackout window — is
Interval List Intersections.
The two-pointer O(m+n) walk over both sorted window
lists replaces an O(m×n) brute-force cross-check
that was timing out on the scheduling dashboard.
On AWS, the same pattern merges CloudWatch
maintenance windows from two separate accounts
to find the combined blackout period before a
cross-account deployment.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra